In [1]:
import pandas as pd

users_df = pd.DataFrame({
    "user_id":[1,2,3,4,5,6,7,8],
    "name":["Alice","Bob","Charlie","Diana","Eve","Frank","Grace","Henry"],
    "country":["UK","France","UK","Germany","France","UK","Germany","UK"],
    "age":[24,30,22,28,35,26,31,29],
    "subscription":["Premium","Free","Premium","Premium","Free","Premium","Free","Premium"]
})

songs_df = pd.DataFrame({
    "song_id":[101,102,103,104,105,106,107],
    "song_name":[
        "Midnight City","Blinding Lights","Numb",
        "Viva La Vida","Levitating","Lose Yourself","Fix You"
    ],
    "artist":[
        "M83","The Weeknd","Linkin Park",
        "Coldplay","Dua Lipa","Eminem","Coldplay"
    ],
    "genre":[
        "Electronic","Pop","Rock",
        "Alternative","Pop","Hip-Hop","Alternative"
    ],
    "duration_sec":[250,200,210,240,203,326,295]
})

listens_df = pd.DataFrame({
    "listen_id":[
        1001,1002,1003,1004,1005,1006,
        1007,1008,1009,1010,1011,1012,
        1013,1014,1015
    ],
    "user_id":[
        1,2,3,1,4,5,
        6,2,3,7,8,1,
        5,6,3
    ],
    "song_id":[
        101,102,103,104,105,101,
        102,103,104,105,106,107,
        104,101,102
    ],
    "listen_date":pd.to_datetime([
        "2024-01-01","2024-01-02","2024-01-02",
        "2024-01-03","2024-01-04","2024-01-04",
        "2024-01-05","2024-01-06","2024-01-07",
        "2024-01-08","2024-01-08","2024-01-09",
        "2024-01-10","2024-01-11","2024-01-12"
    ])
})


In [2]:
'''
Task 1 - Build master listening table
'''

master_df = (
    pd.merge(listens_df,users_df,on="user_id")
    .merge(songs_df, on='song_id')
)
master_df.head()

,listen_id,user_id,song_id,listen_date,name,country,age,subscription,song_name,artist,genre,duration_sec
0,1001,1,101,2024-01-01,Alice,UK,24,Premium,Midnight City,M83,Electronic,250
1,1002,2,102,2024-01-02,Bob,France,30,Free,Blinding Lights,The Weeknd,Pop,200
2,1003,3,103,2024-01-02,Charlie,UK,22,Premium,Numb,Linkin Park,Rock,210
3,1004,1,104,2024-01-03,Alice,UK,24,Premium,Viva La Vida,Coldplay,Alternative,240
4,1005,4,105,2024-01-04,Diana,Germany,28,Premium,Levitating,Dua Lipa,Pop,203


In [3]:
'''
Task 2 - Most played artists
'''
df_most_played_artists = (
    master_df
    .groupby('user_id')
    .agg(number_of_listens = ('listen_id','nunique'))
    .sort_values('number_of_listens',ascending=False)
    .iloc[0:3,:]
)
df_most_played_artists

,number_of_listens
user_id,
1,3
3,3
2,2


In [4]:
'''
Task 3 - Listening time per user
'''
df_most_played_artists = (
    master_df
    .groupby('user_id')
    .agg(total_listening_time = ('duration_sec','sum'))
    .sort_values('total_listening_time',ascending=False)
)
df_most_played_artists

,total_listening_time
user_id,
1,785
3,650
5,490
6,450
2,410
8,326
4,203
7,203


In [5]:
'''
Task 4 - Genre popularity
'''
df_genre_popularity = (
    master_df
    .groupby(['country','genre'])
    .agg(number_of_listens = ('listen_id','nunique'))
    .sort_values('number_of_listens',ascending=False)
)
df_genre_popularity

number_of_listens
country genre                         
UK      Alternative                  3
Germany Pop                          2
UK      Pop                          2
        Electronic                   2
France  Electronic                   1
        Alternative                  1
        Pop                          1
        Rock                         1
UK      Hip-Hop                      1
        Rock                         1

In [6]:
'''
Task 5 - Metrics calculation
'''
df_metrics = (
    master_df
    .loc[lambda x: x.duration_sec > 220]
    .groupby(['genre','subscription'])
    .agg(number_of_listens = ('listen_id','nunique'),
         unique_listeners = ('user_id','nunique'),
         average_song_duration = ('duration_sec','mean')
         )
    .sort_values('number_of_listens',ascending=False)
)
df_metrics

,,number_of_listens,unique_listeners,average_song_duration
genre,subscription,,,
Alternative,Premium,3,2,258.333333
Electronic,Premium,2,2,250.000000
Alternative,Free,1,1,240.000000
Electronic,Free,1,1,250.000000
Hip-Hop,Premium,1,1,326.000000
